# Gaussian Mixture Models (GMM)

This module implements the Expectation-Maximization (EM) algorithm to fit Gaussian Mixture Models from scratch using NumPy. The implementation supports multiple covariance types (`full`, `tied`, `diagonal`, `spherical`) and handles numerical stability using log-space computations.

### 1. The Expectation-Maximization (EM) Algorithm

The GMM attempts to maximize the likelihood of the observed data $X$ by iteratively alternating between two steps:

1. **E-Step (Expectation):** Estimate the probability (responsibility) that each data point belongs to each cluster using current parameters.


2. **M-Step (Maximization):** Update the model parameters ($\pi, \mu, \Sigma$) to maximize the likelihood given the estimated responsibilities.



##### The Objective Function: Log-Likelihood

We maximize the log-likelihood of the data:
$$\ln p(X | \pi, \mu, \Sigma) = \sum_{i=1}^{N} \ln \left( \sum_{k=1}^{K} \pi_k \mathcal{N}(x_i | \mu_k, \Sigma_k) \right)$$

---

## 2. Step-by-Step Mathematical Derivation

### A. The E-Step: Computing Responsibilities

In this step, we calculate , the posterior probability that data point $x_i$ belongs to cluster $k$.

**Formula:**
$$r_{ik} = \frac{\pi_k \mathcal{N}(x_i | \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \mathcal{N}(x_i | \mu_j, \Sigma_j)}$$


**Implementation Detail (Log-Space):**
To prevent numerical underflow (where probabilities become so small they round to zero), we compute everything in log-space using the **Log-Sum-Exp** trick:
$$\ln(r_{ik}) = \ln(\text{weighted\_prob}_{ik}) - \ln\left(\sum_{j} \exp(\ln(\text{weighted\_prob}_{ij}))\right)$$

*Where $\ln(\text{weighted\_prob}_{ik}) = \ln(\pi_k) + \ln \mathcal{N}(x_i | \mu_k, \Sigma_k)$.*

### B. The M-Step: Parameter Updates

We re-estimate parameters based on the effective number of points $N_k$ assigned to each cluster $k$.

**1. Total Weight ($N_k$):**
$$N_k = \sum_{i=1}^{N} r_{ik}$$

**2. Mixing Coefficients (Priors $\pi_k$):**
$$\pi_k = \frac{N_k}{N}$$

**3. Means ($\mu_k$):**
$$\mu_k = \frac{1}{N_k} \sum_{i=1}^{N} r_{ik} x_i$$

**4. Covariances ($\Sigma_k$):**
The update rule depends on the `cov_type` hyperparameter:

* **Full Covariance:** Each cluster has its own general covariance matrix allowing for ellipsoidal clusters of any orientation.
$$\Sigma_k = \frac{1}{N_k} \sum_{i=1}^{N} r_{ik} (x_i - \mu_k)(x_i - \mu_k)^T$$

* **Tied Covariance:** All clusters share the same covariance matrix.
$$\Sigma = \frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{K} r_{ik} (x_i - \mu_k)(x_i - \mu_k)^T$$

* **Diagonal Covariance:** Features are assumed independent; variance is stored per feature per cluster (axis-aligned ellipsoids).
$$\sigma_{k, d}^2 = \frac{1}{N_k} \sum_{i=1}^{N} r_{ik} (x_{i, d} - \mu_{k, d})^2$$

* **Spherical Covariance:** Each cluster has a single variance value (spherical clusters).
$$\sigma_k^2 = \frac{1}{N_k \cdot D} \sum_{i=1}^{N} r_{ik} \| x_i - \mu_k \|^2$$

*Note: A small regularization term (`reg_covar`) is added to the diagonal of covariance matrices to ensure they remain positive semi-definite.*

---

## 3. Multivariate Gaussian Density

The probability density function for a $D$-dimensional Gaussian is:
$$\mathcal{N}(x | \mu, \Sigma) = \frac{1}{\sqrt{(2\pi)^D |\Sigma|}} \exp\left( -\frac{1}{2} (x - \mu)^T \Sigma^{-1} (x - \mu) \right)$$

In our code (`_log_pdf_multivariate_gaussian`), we compute the log of this directly:
$$\ln \mathcal{N}(x) = -\frac{1}{2} \left( D \ln(2\pi) + \ln|\Sigma| + (x - \mu)^T \Sigma^{-1} (x - \mu) \right)$$

*We utilize **Cholesky decomposition** ($\Sigma = L L^T$) to compute the log-determinant ($\ln|\Sigma|$) and solve linear systems efficiently without explicitly inverting the matrix, which improves numerical stability.*

---

## 4. Model Selection Metrics

To determine the optimal number of components and covariance type, we utilize information criteria:

**Bayesian Information Criterion (BIC):**
Penalizes model complexity more heavily to prevent overfitting.
$$\text{BIC} = k \ln(N) - 2 \ln(\hat{L})$$


**Akaike Information Criterion (AIC):**
$$\text{AIC} = 2k - 2 \ln(\hat{L})$$

Where:

* $N$: Number of data samples.
* $\hat{L}$: The maximized log-likelihood.
* $k$: Number of free parameters (calculated based on `cov_type` and input dimensions).